

## 1) silver.tb_avaliacoes_usuarios
DataFrame df <- bronze.tb_movies_reviews.
Bloco abaixo: imports, cria o database silver, le a Bronze e inspeciona os dados.

In [0]:
#Imports e criacao do database da camada Silver
from pyspark.sql import functions as f
from pyspark.sql.window import Window

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

#Le a tabela Bronze de avaliacoes (so leitura, a Bronze nao e alterada)
df= spark.table("bronze.tb_movies_reviews")
df.show(5)

#Expressoes reutilizadas no select: nota como numero e comentario sem espacos
nota = f.col("nota").cast("double")
comentario = f.trim(f.col("comentario"))

#Inspecao: tipos das colunas e valores distintos da nota
df.printSchema()
df.select("nota").distinct().show(500)


+------+--------------------+----+--------------------+--------------------+
|    id|                nome|nota|          comentario|  ingestion_datetime|
+------+--------------------+----+--------------------+--------------------+
|442113| Mariana Cardoso 277| 4.4|                NULL|2026-09-20 21:19:...|
|637007|      Lucas Reis 602| 3.9|                NULL|2026-09-20 21:19:...|
|449479|      Sérgio Freitas| 0.7|Péssimo em todos ...|2026-09-20 21:19:...|
|413036|Gabriela Monteiro...| 7.5|                NULL|2026-09-20 21:19:...|
|528480|   Leonardo Monteiro| 6.3|Assisti até o fin...|2026-09-20 21:19:...|
+------+--------------------+----+--------------------+--------------------+
only showing top 5 rows
root
 |-- id: string (nullable = true)
 |-- nome: string (nullable = true)
 |-- nota: string (nullable = true)
 |-- comentario: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)

+----+
|nota|
+----+
| 4.4|
| 3.9|
| 0.7|
| 7.5|
| 6.3|
| 8.2|
| 4.7|
| 6.5|

DataFrame avaliacoes <- df.
Renomeia as colunas, limita a nota a 0-10, preenche comentario vazio e remove duplicatas.

In [0]:
#Renomeia para PT-BR e aplica as regras de negocio
avaliacoes = df.select(
    f.trim(f.col("id")).alias("id_filme"),
    f.col("nome").alias("nome_usuario"),
    #Nota fora de 0-10 vira NULL (sem otherwise = NULL)
    f.when(nota.between(0, 10), nota).alias("nota_usuario"),
    #Comentario NULL, vazio ou so com espacos vira "Sem comentario"
    f.when(
        (comentario.isNull()) | (comentario == ""),
        f.lit("Sem comentário")
    ).otherwise(comentario).alias("comentario_usuario"),
).dropDuplicates()   #remove duplicatas integrais (filme, usuario, nota, comentario)

avaliacoes.printSchema()
avaliacoes.show(10, truncate=False)

root
 |-- id_filme: string (nullable = true)
 |-- nome_usuario: string (nullable = true)
 |-- nota_usuario: double (nullable = true)
 |-- comentario_usuario: string (nullable = true)

+--------+---------------------+------------+-----------------------------------------------+
|id_filme|nome_usuario         |nota_usuario|comentario_usuario                             |
+--------+---------------------+------------+-----------------------------------------------+
|442113  |Mariana Cardoso 277  |4.4         |Sem comentário                                 |
|637007  |Lucas Reis 602       |3.9         |Sem comentário                                 |
|449479  |Sérgio Freitas       |0.7         |Péssimo em todos os sentidos.                  |
|413036  |Gabriela Monteiro 401|7.5         |Sem comentário                                 |
|528480  |Leonardo Monteiro    |6.3         |Assisti até o final mas não me marcou.         |
|387727  |Maria Alves 707      |8.2         |Gostei bastante, re

Conferencia de avaliacoes (contagens, nulos e faixa da nota) e gravacao em silver.tb_avaliacoes_usuarios.

In [0]:
#Conferencia: linhas antes e depois da deduplicacao
print("linhas na Bronze:", df.count())
print("linhas na Silver (sem duplicatas):", avaliacoes.count())

#Nao deve sobrar comentario nulo ou vazio (esperado: 0)
print("comentários nulos/vazios:",
      avaliacoes.filter(f.col("comentario_usuario").isNull() | (f.col("comentario_usuario") == "")).count())

#Nao deve sobrar nota fora da faixa (esperado: 0)
print("notas fora de 0-10:",
      avaliacoes.filter((f.col("nota_usuario") < 0) | (f.col("nota_usuario") > 10)).count())

#O cast nao deve ter criado nulos novos (os dois numeros devem ser iguais)
print("nulos na Bronze:", df.filter(f.col("nota").isNull()).count())
print("nulos apos cast:", avaliacoes.filter(f.col("nota_usuario").isNull()).count())

#Grava na Silver (overwrite: rodar varias vezes da o mesmo resultado)
avaliacoes.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_avaliacoes_usuarios")

spark.table("silver.tb_avaliacoes_usuarios").show(10, truncate=False)


linhas na Bronze: 32412
linhas na Silver (sem duplicatas): 32412
comentários nulos/vazios: 0
notas fora de 0-10: 0
nulos na Bronze: 1651
nulos apos cast: 1651
+--------+---------------------+------------+-----------------------------------+
|id_filme|nome_usuario         |nota_usuario|comentario_usuario                 |
+--------+---------------------+------------+-----------------------------------+
|637007  |Lucas Reis 602       |3.9         |Sem comentário                     |
|1100094 |Gabriel Carvalho 581 |6.2         |Aceitável, mas esperava mais.      |
|628575  |Alexandre Barbosa 220|0.3         |Péssimo em todos os sentidos.      |
|573249  |Rodrigo Oliveira 273 |0.5         |Péssimo em todos os sentidos.      |
|592539  |Pedro Costa 181      |4.8         |Não gostei, história confusa.      |
|464493  |Adriana Dias 257     |0.4         |Péssimo em todos os sentidos.      |
|1199748 |Eduardo Dias 177     |6.0         |Poderia ser melhor, mas não é ruim.|
|640543  |Cristina Mo

## 2) silver.tb_metricas_engajamento
DataFrame df <- bronze.tb_movies_metrics.
Bloco abaixo: le, inspeciona e mantem so a carga mais recente de cada filme (id).

In [0]:
#Le a Bronze de metricas e inspeciona a sujeira
df = spark.table("bronze.tb_movies_metrics")
df.printSchema()
df.show(20, truncate=False)   #olhe: virgula decimal? textos deslocados?

#Deduplicacao: a Bronze e append, entao fica so a carga mais recente por id
df = df.withColumn("id", f.trim(f.col("id")))
w = Window.partitionBy("id").orderBy(f.col("ingestion_datetime").desc())
df = df.withColumn("linha", f.row_number().over(w)).filter(f.col("linha") == 1).drop("linha")
print("filmes:", df.count())

root
 |-- id: string (nullable = true)
 |-- popularity: string (nullable = true)
 |-- vote_average: string (nullable = true)
 |-- vote_count: string (nullable = true)
 |-- averageRating: string (nullable = true)
 |-- numVotes: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)

+------+----------+------------+----------+-------------+--------+--------------------------+
|id    |popularity|vote_average|vote_count|averageRating|numVotes|ingestion_datetime        |
+------+----------+------------+----------+-------------+--------+--------------------------+
|293660|72.735    |7.606       |28894     |8.0          |1270339 |2026-09-20 21:19:43.038649|
|299536|154,34    |8.255       |27713     |8.4          |1406782 |2026-09-20 21:19:43.038649|
|299534|91.756    |8.263       |23857     |8.4          |1484150 |2026-09-20 21:19:43.038649|
|475557|54,522    |8.168       |23425     |8.3          |1723035 |2026-09-20 21:19:43.038649|
|271110|70.741    |7.4         |2154

Expressoes (colunas, nao DataFrames) que limpam o texto e convertem para numero:
popularidade, nota_tmdb, nota_imdb, votos_tmdb e votos_imdb.

In [0]:
#Popularidade: troca virgula por ponto e so converte se parecer numero
#(texto deslocado do column shift vira NULL em vez de quebrar)
pop_texto = f.regexp_replace(f.trim(f.col("popularity")), ",", ".")
popularidade = f.when(pop_texto.rlike(r"^-?[0-9]+(\.[0-9]+)?$"), pop_texto.cast("double"))

#Notas e votos: mesma ideia (so converte se parecer numero)
nota_tmdb_texto = f.regexp_replace(f.trim(f.col("vote_average")), ",", ".")
nota_tmdb = f.when(nota_tmdb_texto.rlike(r"^-?[0-9]+(\.[0-9]+)?$"), nota_tmdb_texto.cast("double"))

nota_imdb_texto = f.regexp_replace(f.trim(f.col("averageRating")), ",", ".")
nota_imdb = f.when(nota_imdb_texto.rlike(r"^-?[0-9]+(\.[0-9]+)?$"), nota_imdb_texto.cast("double"))

#Contagens de votos: so inteiros
votos_tmdb_texto = f.trim(f.col("vote_count"))
votos_tmdb = f.when(votos_tmdb_texto.rlike(r"^-?[0-9]+(\.0+)?$"), votos_tmdb_texto.cast("double"))

votos_imdb_texto = f.trim(f.col("numVotes"))
votos_imdb = f.when(votos_imdb_texto.rlike(r"^-?[0-9]+(\.0+)?$"), votos_imdb_texto.cast("double"))


DataFrame metricas <- df.
Renomeia, aplica as regras (nota 0-10, negativos -> NULL), confere os nulos e grava em silver.tb_metricas_engajamento.

In [0]:
#Monta a tabela final com nomes em PT-BR e as regras de negocio
metricas = df.select(
    f.col("id").alias("id_filme"),
    f.when(popularidade >= 0, popularidade).alias("popularidade"),              #negativo = NULL
    f.when(nota_tmdb.between(0, 10), nota_tmdb).alias("nota_media_tmdb"),       #fora de 0-10 = NULL
    f.when(votos_tmdb >= 0, votos_tmdb).cast("int").alias("qtd_votos_tmdb"),    #negativo = NULL
    f.when(nota_imdb.between(0, 10), nota_imdb).alias("nota_media_imdb"),       #fora de 0-10 = NULL
    f.when(votos_imdb >= 0, votos_imdb).cast("int").alias("qtd_votos_imdb"),    #negativo = NULL
)
metricas.printSchema()

#Conferencia: nulos por coluna e estatisticas
metricas.select([f.count(f.when(f.col(c).isNull(), 1)).alias(c) for c in metricas.columns]).show()
metricas.describe().show()

#Grava na Silver
metricas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_metricas_engajamento")
spark.table("silver.tb_metricas_engajamento").show(5, truncate=False)

root
 |-- id_filme: string (nullable = true)
 |-- popularidade: double (nullable = true)
 |-- nota_media_tmdb: double (nullable = true)
 |-- qtd_votos_tmdb: integer (nullable = true)
 |-- nota_media_imdb: double (nullable = true)
 |-- qtd_votos_imdb: integer (nullable = true)

+--------+------------+---------------+--------------+---------------+--------------+
|id_filme|popularidade|nota_media_tmdb|qtd_votos_tmdb|nota_media_imdb|qtd_votos_imdb|
+--------+------------+---------------+--------------+---------------+--------------+
|       0|        4168|           3519|          8036|          13197|         11574|
+--------+------------+---------------+--------------+---------------+--------------+

+-------+-----------------+-----------------+------------------+-----------------+-----------------+------------------+
|summary|         id_filme|     popularidade|   nota_media_tmdb|   qtd_votos_tmdb|  nota_media_imdb|    qtd_votos_imdb|
+-------+-----------------+-----------------+------

## 3) silver.tb_cotacao_dolar
DataFrame df <- bronze.tb_cotacao_dolar.
Bloco abaixo: le a cotacao do dolar e inspeciona.

In [0]:
#Le a Bronze da cotacao (veio da API do Banco Central)
df = spark.table("bronze.tb_cotacao_dolar")
df.printSchema()
df.show(10, truncate=False)

root
 |-- cotacaoCompra: double (nullable = true)
 |-- dataHoraCotacao: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)

+-------------+--------------------------+--------------------------+
|cotacaoCompra|dataHoraCotacao           |ingestion_datetime        |
+-------------+--------------------------+--------------------------+
|5.169        |2026-09-14 13:10:08.144425|2026-09-20 21:19:58.819342|
|5.1484       |2026-09-15 13:09:19.199664|2026-09-20 21:19:58.819342|
|5.152        |2026-09-16 13:05:30.35873 |2026-09-20 21:19:58.819342|
|5.1515       |2026-09-17 13:03:21.858212|2026-09-20 21:19:58.819342|
|5.1569       |2026-09-18 13:03:34.742036|2026-09-20 21:19:58.819342|
+-------------+--------------------------+--------------------------+



DataFrames cot e calendario.
cot: converte os tipos e mantem 1 cotacao por dia. calendario: uma linha para cada dia do periodo, ate hoje.

In [0]:
#Converte tipos: data (10 primeiros caracteres) e cotacao como numero
cot = df.select(
    f.to_date(f.substring(f.col("dataHoraCotacao"), 1, 10)).alias("data_cotacao"),
    f.col("cotacaoCompra").cast("double").alias("cotacao_compra"),
    f.col("ingestion_datetime"),
).filter(f.col("data_cotacao").isNotNull())

#Deduplicacao: fica 1 cotacao por dia (a carga mais recente)
w_dia = Window.partitionBy("data_cotacao").orderBy(f.col("ingestion_datetime").desc())
cot = cot.withColumn("linha", f.row_number().over(w_dia)).filter(f.col("linha") == 1).drop("linha", "ingestion_datetime")
cot.orderBy("data_cotacao").show()

#Calendario continuo: do primeiro dia ate hoje (cobre fins de semana e feriados)
limites = cot.agg(f.min("data_cotacao").alias("inicio"), f.max("data_cotacao").alias("fim"))
calendario = limites.select(
    f.explode(f.sequence(f.col("inicio"), f.greatest(f.col("fim"), f.current_date()), f.expr("INTERVAL 1 DAY"))).alias("data_cotacao")
)
calendario.show()

+------------+--------------+
|data_cotacao|cotacao_compra|
+------------+--------------+
|  2026-09-14|         5.169|
|  2026-09-15|        5.1484|
|  2026-09-16|         5.152|
|  2026-09-17|        5.1515|
|  2026-09-18|        5.1569|
+------------+--------------+

+------------+
|data_cotacao|
+------------+
|  2026-09-14|
|  2026-09-15|
|  2026-09-16|
|  2026-09-17|
|  2026-09-18|
|  2026-09-19|
|  2026-09-20|
|  2026-09-21|
+------------+



DataFrame cotacao <- calendario + cot.
Junta os dois, aplica o forward fill nos dias sem cotacao e grava em silver.tb_cotacao_dolar.

In [0]:
#Junta o calendario com as cotacoes (dias sem cotacao ficam NULL)
cotacao = calendario.join(cot, on="data_cotacao", how="left")

#Forward fill: dia sem cotacao recebe o valor do ultimo dia util disponivel
w_ff = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, 0)
cotacao = cotacao.withColumn("cotacao_compra", f.last("cotacao_compra", ignorenulls=True).over(w_ff))
cotacao.orderBy("data_cotacao").show(30)

#Grava na Silver
cotacao.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_cotacao_dolar")
spark.table("silver.tb_cotacao_dolar").orderBy("data_cotacao").show(10)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------------+--------------+
|data_cotacao|cotacao_compra|
+------------+--------------+
|  2026-09-14|         5.169|
|  2026-09-15|        5.1484|
|  2026-09-16|         5.152|
|  2026-09-17|        5.1515|
|  2026-09-18|        5.1569|
|  2026-09-19|        5.1569|
|  2026-09-20|        5.1569|
|  2026-09-21|        5.1569|
+------------+--------------+

+------------+--------------+
|data_cotacao|cotacao_compra|
+------------+--------------+
|  2026-09-14|         5.169|
|  2026-09-15|        5.1484|
|  2026-09-16|         5.152|
|  2026-09-17|        5.1515|
|  2026-09-18|        5.1569|
|  2026-09-19|        5.1569|
|  2026-09-20|        5.1569|
|  2026-09-21|        5.1569|
+------------+--------------+



## 4) silver.tb_financeiro_filmes
DataFrame df <- bronze.tb_movies_financials.
Bloco abaixo: le, deduplica por id e define as expressoes orcamento e receita (texto limpo e convertido).

In [0]:
#Le a Bronze financeira e inspeciona (veja: "Unknown", "$", pontos de milhar, zeros)
df = spark.table("bronze.tb_movies_financials")
df.printSchema()
df.show(20, truncate=False)

#Deduplicacao: uma versao por filme (carga mais recente)
df = df.withColumn("id", f.trim(f.col("id")))
w = Window.partitionBy("id").orderBy(f.col("ingestion_datetime").desc())
df = df.withColumn("linha", f.row_number().over(w)).filter(f.col("linha") == 1).drop("linha")

#Orcamento: tira simbolos de moeda, espacos, virgulas e pontos de milhar.
#Texto como "Unknown" nao passa no rlike e vira NULL.
orc_texto = f.regexp_replace(f.trim(f.col("budget")), r"USD|BRL|R\$|\$|\s", "")
orc_texto = f.regexp_replace(orc_texto, ",", "")
orc_texto = f.regexp_replace(orc_texto, r"\.(?=[0-9]{3})", "")
orcamento = f.when(orc_texto.rlike(r"^-?[0-9]+(\.[0-9]+)?$"), orc_texto.cast("double"))

#Receita: mesma limpeza
rec_texto = f.regexp_replace(f.trim(f.col("revenue")), r"USD|BRL|R\$|\$|\s", "")
rec_texto = f.regexp_replace(rec_texto, ",", "")
rec_texto = f.regexp_replace(rec_texto, r"\.(?=[0-9]{3})", "")
receita = f.when(rec_texto.rlike(r"^-?[0-9]+(\.[0-9]+)?$"), rec_texto.cast("double"))

root
 |-- id: string (nullable = true)
 |-- budget: string (nullable = true)
 |-- revenue: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)

+------+-----------+-------------+--------------------------+
|id    |budget     |revenue      |ingestion_datetime        |
+------+-----------+-------------+--------------------------+
|293660|58000000   |Unknown      |2026-09-20 21:19:38.911217|
|299536|300000000  |2052415039   |2026-09-20 21:19:38.911217|
|299534|356000000  |2800000000   |2026-09-20 21:19:38.911217|
|475557|55000000   |1074458282   |2026-09-20 21:19:38.911217|
|271110|250000000  |Não Informado|2026-09-20 21:19:38.911217|
|284054|200000000  |1349926083   |2026-09-20 21:19:38.911217|
|284052|180000000  |676343174    |2026-09-20 21:19:38.911217|
|315635|175000000  |880166924    |2026-09-20 21:19:38.911217|
|283995|200000000  |863756051    |2026-09-20 21:19:38.911217|
|297761|175000000  |746846894    |2026-09-20 21:19:38.911217|
|284053|180000000  |8553

DataFrame financeiro <- df.
Busca a taxa do dolar, monta orcamento e receita em USD e calcula os valores em BRL e o lucro.

In [0]:
#Taxa unica: a cotacao mais recente da serie (a API traz so os ultimos 7 dias)
taxa = spark.table("silver.tb_cotacao_dolar").orderBy(f.col("data_cotacao").desc()).first()["cotacao_compra"]
print("Taxa USD->BRL:", taxa)

#Valores em USD: zero ou negativo vira NULL
financeiro = df.select(
    f.col("id").alias("id_filme"),
    f.when(orcamento > 0, orcamento).cast("decimal(18,2)").alias("orcamento_usd"),
    f.when(receita > 0, receita).cast("decimal(18,2)").alias("receita_usd"),
)

#Conversao para reais com a taxa
financeiro = financeiro.withColumn("orcamento_brl", (f.col("orcamento_usd") * f.lit(taxa)).cast("decimal(18,2)"))
financeiro = financeiro.withColumn("receita_brl", (f.col("receita_usd") * f.lit(taxa)).cast("decimal(18,2)"))

#Lucro = receita - orcamento (se algum for NULL, o lucro fica NULL)
financeiro = financeiro.withColumn("lucro_usd", (f.col("receita_usd") - f.col("orcamento_usd")).cast("decimal(18,2)"))
financeiro = financeiro.withColumn("lucro_brl", (f.col("receita_brl") - f.col("orcamento_brl")).cast("decimal(18,2)"))


Taxa USD->BRL: 5.1569


Calcula a margem de lucro %, confere o DataFrame financeiro e grava em silver.tb_financeiro_filmes.

In [0]:
#Margem % = lucro / receita * 100, so quando receita > 0 (evita divisao por zero)
financeiro = financeiro.withColumn(
    "margem_lucro_pct",
    f.when(f.col("receita_usd") > 0, f.round(f.col("lucro_usd") / f.col("receita_usd") * 100, 2))
)
financeiro.printSchema()

#Conferencia: amostra e nulos por coluna
financeiro.show(10, truncate=False)
financeiro.select([f.count(f.when(f.col(c).isNull(), 1)).alias(c) for c in financeiro.columns]).show()

#Grava na Silver
financeiro.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_financeiro_filmes")
spark.table("silver.tb_financeiro_filmes").show(5, truncate=False)


root
 |-- id_filme: string (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- margem_lucro_pct: decimal(25,2) (nullable = true)

+--------+-------------+-----------+-------------+-----------+---------+---------+----------------+
|id_filme|orcamento_usd|receita_usd|orcamento_brl|receita_brl|lucro_usd|lucro_brl|margem_lucro_pct|
+--------+-------------+-----------+-------------+-----------+---------+---------+----------------+
|1000004 |NULL         |NULL       |NULL         |NULL       |NULL     |NULL     |NULL            |
|1000005 |NULL         |NULL       |NULL         |NULL       |NULL     |NULL     |NULL            |
|1000007 |NULL         |NULL       |NULL         |NULL       |NULL     |NULL     |NULL            |

## 5) silver.tb_info_filmes
DataFrame df <- bronze.tb_movies_info.
Bloco abaixo: le, inspeciona status e release_date, deduplica por id e normaliza o texto do status (st).

In [0]:
#Le a Bronze de info e inspeciona status e datas (varios formatos)
df = spark.table("bronze.tb_movies_info")
df.printSchema()
df.select("status").distinct().show(50, truncate=False)
df.select("release_date").distinct().show(40, truncate=False)

#Deduplicacao: um registro por filme, mantendo a carga mais recente
df = df.withColumn("id", f.trim(f.col("id")))
w = Window.partitionBy("id").orderBy(f.col("ingestion_datetime").desc())
df = df.withColumn("linha", f.row_number().over(w)).filter(f.col("linha") == 1).drop("linha")
print("filmes:", df.count())

#Status - passo 1: normaliza (minusculas, hifen vira espaco, sem ruido)
st = f.lower(f.trim(f.col("status")))
st = f.regexp_replace(st, r"[-_]+", " ")
st = f.regexp_replace(st, r"[^a-z ]", "")
st = f.trim(f.regexp_replace(st, r"\s+", " "))

root
 |-- id: string (nullable = true)
 |-- tconst: string (nullable = true)
 |-- title: string (nullable = true)
 |-- original_title: string (nullable = true)
 |-- original_language: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- runtime: string (nullable = true)
 |-- status: string (nullable = true)
 |-- overview: string (nullable = true)
 |-- tagline: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)

+---------------------------------------------------------------------------+
|status                                                                     |
+---------------------------------------------------------------------------+
|Released                                                                   |
|released                                                                   |
|RELEASED                                                                   |
| his errant dad returns                                             

Expressoes status (traducao para portugues) e data (converte varios formatos de data).

In [0]:
#Status - passo 2: traduz; o que nao mapear vira "Nao Informado"
status = (
    f.when(st == "released", "Lançado")
     .when(st.isin("post production", "postproduction"), "Pós-Produção")
     .when(st.isin("in production", "inproduction"), "Em Produção")
     .when(st == "planned", "Planejado")
     .when(st == "rumored", "Rumores")
     .when(st.isin("canceled", "cancelled"), "Cancelado")
     .otherwise("Não Informado")
)

#Data multi-formato: tenta cada padrao (try_ devolve NULL em vez de quebrar)
#e o coalesce fica com o primeiro que funcionar. A ordem importa nas datas ambiguas.
data = f.coalesce(
    f.expr("try_to_timestamp(trim(release_date), 'yyyy-MM-dd')").cast("date"),
    f.expr("try_to_timestamp(trim(release_date), 'yyyy-MM-dd HH:mm:ss')").cast("date"),
    f.expr("try_to_timestamp(trim(release_date), 'M-d-yyyy')").cast("date"),
    f.expr("try_to_timestamp(trim(release_date), 'd-M-yyyy')").cast("date"),
    f.expr("try_to_timestamp(trim(release_date), 'yyyy/M/d')").cast("date"),
    f.expr("try_to_timestamp(trim(release_date), 'M/d/yyyy')").cast("date"),
    f.expr("try_to_timestamp(trim(release_date), 'd/M/yyyy')").cast("date"),
    f.expr("try_to_timestamp(trim(release_date), 'yyyyMMdd')").cast("date"),
)

DataFrame info <- df.
Monta a tabela final, cria ano_lancamento, confere e grava em silver.tb_info_filmes.

In [0]:
#Duracao: so converte se for numero inteiro
duracao_texto = f.trim(f.col("runtime"))
duracao = f.when(duracao_texto.rlike(r"^[0-9]+(\.0+)?$"), duracao_texto.cast("double"))

#Monta a tabela final com nomes em PT-BR
info = df.select(
    f.col("id").alias("id_filme"),
    f.trim(f.col("title")).alias("titulo"),
    f.trim(f.col("original_title")).alias("titulo_original"),
    data.alias("data_lancamento"),
    duracao.cast("int").alias("duracao_minutos"),
    f.col("original_language").alias("idioma_original"),
    status.alias("status_filme"),
    f.col("overview").alias("sinopse"),
    f.col("tagline").alias("frase_divulgacao"),
)
#Coluna derivada: ano extraido da data de lancamento
info = info.withColumn("ano_lancamento", f.year(f.col("data_lancamento")))
info.printSchema()

#Conferencia: status traduzidos, datas que nao converteram e unicidade por filme
info.groupBy("status_filme").count().show()
info.filter(f.col("data_lancamento").isNull()).show(10, truncate=False)
print("filmes:", info.count(), "| ids distintos:", info.select("id_filme").distinct().count())

#Grava na Silver
info.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_info_filmes")
spark.table("silver.tb_info_filmes").show(5, truncate=False)


root
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = false)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)
 |-- ano_lancamento: integer (nullable = true)

+-------------+-----+
| status_filme|count|
+-------------+-----+
|      Lançado|96463|
|  Em Produção|  604|
| Pós-Produção|  701|
|Não Informado|   64|
|    Planejado|   47|
+-------------+-----+

+--------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## 6) silver.tb_generos
DataFrame credits <- bronze.tb_credits_and_tags.
Bloco abaixo: le e mantem so a carga mais recente por id. O credits alimenta as tabelas de generos e de pessoas/empresas.

In [0]:
#Le a Bronze de creditos e tags
df = spark.table("bronze.tb_credits_and_tags")
df.printSchema()
df.show(5, truncate=False)

#Deduplicacao: uma versao por filme (carga mais recente)
df = df.withColumn("id", f.trim(f.col("id")))
w = Window.partitionBy("id").orderBy(f.col("ingestion_datetime").desc())
credits = df.withColumn("linha", f.row_number().over(w)).filter(f.col("linha") == 1).drop("linha")

root
 |-- id: string (nullable = true)
 |-- genres: string (nullable = true)
 |-- production_companies: string (nullable = true)
 |-- production_countries: string (nullable = true)
 |-- spoken_languages: string (nullable = true)
 |-- keywords: string (nullable = true)
 |-- directors: string (nullable = true)
 |-- writers: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- ingestion_datetime: timestamp (nullable = true)

+------+----------------------------------+--------------------------------------------------------------------------------------+--------------------------------+----------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------------------+------------------------

DataFrame generos <- credits (coluna genres).
Separa e explode os generos, limpa, filtra o lixo, confere e grava em silver.tb_generos.

In [0]:
#Passo 1: uniformiza separadores (; e | viram virgula) e separa em lista
generos_lista = f.split(f.regexp_replace(f.col("genres"), r"[;|]", ","), ",")

#Passo 2: explode (uma linha por genero)
generos = credits.select(f.col("id").alias("id_filme"), f.explode(generos_lista).alias("nome_genero"))

#Passo 3: limpa espacos e padroniza a caixa
generos = generos.withColumn("nome_genero", f.initcap(f.trim(f.regexp_replace(f.col("nome_genero"), r"\s+", " "))))

#Passo 4: remove lixo do column shift (vazio, numero, texto longo, algo com digito)
generos = generos.filter(
    (f.length(f.col("nome_genero")) >= 2)
    & (f.length(f.col("nome_genero")) <= 30)
    & (~f.col("nome_genero").rlike(r"\d"))
)
generos = generos.dropDuplicates()

#Conferencia: contagem por genero (sobrou lixo? aperte o filtro)
generos.groupBy("nome_genero").count().orderBy(f.desc("count")).show(60, truncate=False)

#Grava na Silver
generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_generos")
spark.table("silver.tb_generos").show(5, truncate=False)

+------------------------------+-----+
|nome_genero                   |count|
+------------------------------+-----+
|Drama                         |32646|
|Documentary                   |19233|
|Comedy                        |18821|
|Thriller                      |10400|
|Horror                        |9853 |
|Romance                       |7712 |
|Action                        |6100 |
|Crime                         |4781 |
|Animation                     |4516 |
|Tv Movie                      |4112 |
|Science Fiction               |3808 |
|Family                        |3761 |
|Mystery                       |3355 |
|Fantasy                       |3306 |
|Adventure                     |2889 |
|Music                         |2824 |
|History                       |2439 |
|War                           |974  |
|Western                       |417  |
|Comedy"                       |39   |
|Drama"                        |35   |
|Documentary"                  |15   |
|Horror"                 

## 7) silver.tb_pessoas_empresas
DataFrame atores <- credits (coluna cast).
Separa e explode os nomes, limpa, filtra o lixo e cria tipo_entidade = "Ator".

In [0]:
#---- ATORES ----
#Separa a lista de nomes e explode (uma linha por ator)
atores = credits.select(
    f.col("id").alias("id_filme"),
    f.explode(f.split(f.regexp_replace(f.col("cast"), r"[;|]", ","), ",")).alias("nome_entidade"),
)
#Limpa espacos e padroniza a caixa
atores = atores.withColumn("nome_entidade", f.initcap(f.trim(f.regexp_replace(f.col("nome_entidade"), r"\s+", " "))))
#Remove lixo (tamanho, digitos, frase longa) e marca o tipo
atores = atores.filter(
    (f.length("nome_entidade") >= 2) & (f.length("nome_entidade") <= 80)
    & (~f.col("nome_entidade").rlike(r"\d"))
    & (f.size(f.split(f.col("nome_entidade"), " ")) <= 8)
).withColumn("tipo_entidade", f.lit("Ator"))

DataFrame diretores <- credits (coluna directors).
Mesmo processo dos atores, com tipo_entidade = "Diretor".

In [0]:
#---- DIRETORES ----
#Separa e explode (uma linha por diretor)
diretores = credits.select(
    f.col("id").alias("id_filme"),
    f.explode(f.split(f.regexp_replace(f.col("directors"), r"[;|]", ","), ",")).alias("nome_entidade"),
)
#Limpa espacos e padroniza a caixa
diretores = diretores.withColumn("nome_entidade", f.initcap(f.trim(f.regexp_replace(f.col("nome_entidade"), r"\s+", " "))))
#Remove lixo e marca o tipo
diretores = diretores.filter(
    (f.length("nome_entidade") >= 2) & (f.length("nome_entidade") <= 80)
    & (~f.col("nome_entidade").rlike(r"\d"))
    & (f.size(f.split(f.col("nome_entidade"), " ")) <= 8)
).withColumn("tipo_entidade", f.lit("Diretor"))

DataFrame roteiristas <- credits (coluna writers).
Mesmo processo, com tipo_entidade = "Roteirista".

In [0]:
#---- ROTEIRISTAS ----
#Separa e explode (uma linha por roteirista)
roteiristas = credits.select(
    f.col("id").alias("id_filme"),
    f.explode(f.split(f.regexp_replace(f.col("writers"), r"[;|]", ","), ",")).alias("nome_entidade"),
)
#Limpa espacos e padroniza a caixa
roteiristas = roteiristas.withColumn("nome_entidade", f.initcap(f.trim(f.regexp_replace(f.col("nome_entidade"), r"\s+", " "))))
#Remove lixo e marca o tipo
roteiristas = roteiristas.filter(
    (f.length("nome_entidade") >= 2) & (f.length("nome_entidade") <= 80)
    & (~f.col("nome_entidade").rlike(r"\d"))
    & (f.size(f.split(f.col("nome_entidade"), " ")) <= 8)
).withColumn("tipo_entidade", f.lit("Roteirista"))

DataFrame produtoras <- credits (coluna production_companies).
Mesmo processo, mas aceitando digitos no nome (ex.: "20th Century Fox"), com tipo_entidade = "Produtora".

In [0]:
#---- PRODUTORAS ----
#Separa e explode (uma linha por produtora)
produtoras = credits.select(
    f.col("id").alias("id_filme"),
    f.explode(f.split(f.regexp_replace(f.col("production_companies"), r"[;|]", ","), ",")).alias("nome_entidade"),
)
#Limpa espacos e padroniza a caixa
produtoras = produtoras.withColumn("nome_entidade", f.initcap(f.trim(f.regexp_replace(f.col("nome_entidade"), r"\s+", " "))))
#Remove lixo e marca o tipo (aqui digitos sao permitidos; so rejeita texto so de numero/pontuacao)
produtoras = produtoras.filter(
    (f.length("nome_entidade") >= 2) & (f.length("nome_entidade") <= 80)
    & (~f.col("nome_entidade").rlike(r"^[0-9\W_]+$"))
    & (f.size(f.split(f.col("nome_entidade"), " ")) <= 8)
).withColumn("tipo_entidade", f.lit("Produtora"))

DataFrame pessoas = uniao de atores, diretores, roteiristas e produtoras.
Remove duplicatas, confere e grava em silver.tb_pessoas_empresas.

In [0]:
#Junta os quatro tipos numa dimensao unica e remove duplicatas
pessoas = atores.unionByName(diretores).unionByName(roteiristas).unionByName(produtoras).dropDuplicates()

#Conferencia: contagem por tipo e amostra aleatoria para achar lixo
pessoas.groupBy("tipo_entidade").count().show()
pessoas.orderBy(f.rand()).show(20, truncate=False)

#Grava na Silver
pessoas.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("silver.tb_pessoas_empresas")
spark.table("silver.tb_pessoas_empresas").show(5, truncate=False)

+-------------+------+
|tipo_entidade| count|
+-------------+------+
|      Diretor|107678|
|   Roteirista|137070|
|    Produtora|119581|
|         Ator|541827|
+-------------+------+

+--------+----------------------------+-------------+
|id_filme|nome_entidade               |tipo_entidade|
+--------+----------------------------+-------------+
|794699  |Jonathan Benton             |Ator         |
|498687  |Josie Leung                 |Ator         |
|828613  |Vincent Newman Entertainment|Produtora    |
|1002450 |Imoh Eboh                   |Ator         |
|1098424 |Ramasimhan                  |Diretor      |
|522570  |Alex Pusineri               |Ator         |
|780962  |Léo Fontaine                |Roteirista   |
|727429  |Rai                         |Produtora    |
|727895  |Cenk Köksal                 |Roteirista   |
|622841  |Arci Muñoz                  |Ator         |
|424834  |Erik Reese                  |Roteirista   |
|577060  |Baldr                       |Produtora    |
|7084

## Resumo final
Conta as linhas das 7 tabelas Silver gravadas.

In [0]:
#Contagem de linhas de cada tabela Silver
print("avaliacoes:", spark.table("silver.tb_avaliacoes_usuarios").count())
print("metricas:  ", spark.table("silver.tb_metricas_engajamento").count())
print("cotacao:   ", spark.table("silver.tb_cotacao_dolar").count())
print("financeiro:", spark.table("silver.tb_financeiro_filmes").count())
print("info:      ", spark.table("silver.tb_info_filmes").count())
print("generos:   ", spark.table("silver.tb_generos").count())
print("pessoas:   ", spark.table("silver.tb_pessoas_empresas").count())


avaliacoes: 32412
metricas:   99013
cotacao:    8
financeiro: 99006
info:       97879
generos:    142766
pessoas:    906156
